In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
class PositionEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionEmbedding, self).__init__()
        self.d_model = d_model
        self.freq = torch.exp(-torch.log(1000000.0) * torch.arange(0, d_model, 2).float() / d_model)
        self.position = torch.arange(0, max_len).unsqueeze(1)
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(self.position * self.freq)
        self.pe[:, 1::2] = torch.cos(self.position * self.freq)
        self.pe = self.pe.unsqueeze(0)
        self.register_buffer('pe', self.pe)
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        output = torch.matmul(attn, v)
        return output, attn

    def forward(self, x, mask=None):
        batch_size = x.size(0)

        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        output, attn = self.scaled_dot_product_attention(Q, K, V, mask)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.w_o(output)
        return output, attn


In [1]:
import mujoco
import mujoco.viewer
import numpy as np
import os

# 模型文件路径（相对于 notebook 的位置）
# 如果 notebook 在项目根目录，可以直接使用文件名
model_path = 'scene.xml'  # 或者使用 'panda.xml' 来加载不带场景的机器人

# 加载模型
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

print(f"模型加载成功！")
print(f"自由度 (nq): {model.nq}")
print(f"执行器数量 (nu): {model.nu}")
print(f"执行器名称: {[mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(model.nu)]}")


模型加载成功！
自由度 (nq): 9
执行器数量 (nu): 8
执行器名称: ['actuator1', 'actuator2', 'actuator3', 'actuator4', 'actuator5', 'actuator6', 'actuator7', 'actuator8']


In [2]:
# 重置到初始状态
mujoco.mj_resetData(model, data)

# 尝试使用 keyframe 'home'（如果存在）
home_key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'home')
if home_key_id >= 0:
    mujoco.mj_resetDataKeyframe(model, data, home_key_id)
    print("使用 'home' keyframe 初始化")
else:
    print("使用默认初始化")

# 设置初始控制命令（7个关节 + 1个夹爪）
# actuator1-7: 7个关节的控制（弧度）
# actuator8: 夹爪控制（0-255，0=张开，255=闭合）
ctrl = np.array([0.0, 0.0, 0.0, -1.57, 0.0, 1.57, -0.785, 128])  # 中间位置
data.ctrl[:] = ctrl

print(f"初始控制命令: {ctrl}")
print(f"初始关节位置: {data.qpos[:7]}")


使用 'home' keyframe 初始化
初始控制命令: [  0.      0.      0.     -1.57    0.      1.57   -0.785 128.   ]
初始关节位置: [ 0.       0.       0.      -1.57079  0.       1.57079 -0.7853 ]


In [7]:
# 使用 viewer 进行交互式可视化（在一个单独的窗口中）
# 注意：在 Jupyter notebook 中，viewer 会打开一个新窗口
# 运行这个 cell 会打开可视化窗口，可以在窗口中实时查看和控制机器人

with mujoco.viewer.launch_passive(model, data) as viewer:
    # 设置相机参数
    viewer.cam.lookat[:] = [0.3, 0, 0.4]  # 看向机器人中心
    viewer.cam.distance = 2.0  # 相机距离
    viewer.cam.azimuth = 120  # 方位角
    viewer.cam.elevation = -20  # 仰角
    
    # 运行模拟（手动控制或循环控制）
    # 这里是一个简单的示例，让机器人移动到不同位置
    step = 0
    while viewer.is_running() and step < 10000000:
        step_start = data.time
        
        # 示例：正弦波运动控制
        t = data.time
        ctrl = np.array([
            0.5 * np.sin(t),      # joint1
            0.3 * np.sin(t * 0.8),  # joint2
            0.4 * np.sin(t * 0.6),  # joint3
            -1.57 + 0.5 * np.sin(t * 0.7),  # joint4
            0.2 * np.sin(t * 0.9),  # joint5
            1.57 + 0.3 * np.sin(t * 0.5),  # joint6
            -0.785 + 0.4 * np.sin(t * 0.8),  # joint7
            128 + 50 * np.sin(t * 0.3)  # 夹爪 (78-178)
        ])
        data.ctrl[:] = np.clip(ctrl, model.actuator_ctrlrange[:, 0], model.actuator_ctrlrange[:, 1])
        
        # 执行一步仿真
        mujoco.mj_step(model, data)
        
        # 同步 viewer（限制帧率）
        viewer.sync()
        
        step += 1


In [ ]:
# 键盘控制机械臂笛卡尔位置
# 使用键盘控制末端执行器的位置

import threading
import time
from collections import defaultdict

# 尝试导入键盘库
try:
    import keyboard
    KEYBOARD_AVAILABLE = True
except ImportError:
    KEYBOARD_AVAILABLE = False
    print("提示: 安装 'keyboard' 库可以获得更好的键盘控制体验")
    print("运行: pip install keyboard")
    print("如果没有安装，将使用替代方案")

# 重置数据
mujoco.mj_resetData(model, data)
home_key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'home')
if home_key_id >= 0:
    mujoco.mj_resetDataKeyframe(model, data, home_key_id)

# 获取 hand body 的 id
hand_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'hand')
if hand_body_id < 0:
    print("警告: 未找到 'hand' body，尝试使用 'panda_hand'")
    hand_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'panda_hand')

# 初始化：获取当前末端执行器位置作为目标
mujoco.mj_forward(model, data)
if hand_body_id >= 0:
    target_pos = data.xpos[hand_body_id].copy()
else:
    target_pos = np.array([0.5, 0.0, 0.5])
    print("警告: 无法找到末端执行器，使用默认位置")

# 键盘状态（用于检测按键）
keys_pressed = defaultdict(bool)

# 控制参数
move_speed = 0.015  # 每次移动的距离（米）
ik_gain = 0.15  # 逆运动学增益

def solve_ik(model, data, target_pos):
    """使用雅可比矩阵求解逆运动学"""
    if hand_body_id < 0:
        return False
    
    # 获取当前末端执行器位置
    mujoco.mj_forward(model, data)
    current_pos = data.xpos[hand_body_id]
    
    # 计算位置误差
    pos_error = target_pos - current_pos
    
    # 如果误差很小，认为已到达目标
    if np.linalg.norm(pos_error) < 0.0005:
        return True
    
    # 获取雅可比矩阵（位置部分，3x7）
    jacp = np.zeros((3, model.nv))
    mujoco.mj_jacBodyCom(model, data, jacp, None, hand_body_id)
    
    # 只取前7个关节的雅可比（7自由度）
    jacp = jacp[:, :7]
    
    # 使用伪逆求解关节速度
    jacp_pinv = np.linalg.pinv(jacp)
    dq = jacp_pinv @ pos_error
    
    # 应用增益并更新关节位置
    dq *= ik_gain
    data.qpos[:7] += dq
    
    # 限制关节角度
    for i in range(7):
        if model.jnt_range[i, 0] < model.jnt_range[i, 1]:
            qmin = model.jnt_range[i, 0]
            qmax = model.jnt_range[i, 1]
        else:
            qmin = -np.pi
            qmax = np.pi
        data.qpos[i] = np.clip(data.qpos[i], qmin, qmax)
    
    return False

# 键盘控制函数
def update_target_from_keys():
    """根据按键更新目标位置"""
    global target_pos
    
    if keys_pressed['w'] or keys_pressed['W']:  # 向前（X+）
        target_pos[0] += move_speed
    if keys_pressed['s'] or keys_pressed['S']:  # 向后（X-）
        target_pos[0] -= move_speed
    if keys_pressed['a'] or keys_pressed['A']:  # 向左（Y-）
        target_pos[1] -= move_speed
    if keys_pressed['d'] or keys_pressed['D']:  # 向右（Y+）
        target_pos[1] += move_speed
    if keys_pressed['q'] or keys_pressed['Q']:  # 向上（Z+）
        target_pos[2] += move_speed
    if keys_pressed['e'] or keys_pressed['E']:  # 向下（Z-）
        target_pos[2] -= move_speed
    if keys_pressed['r'] or keys_pressed['R']:  # 重置到初始位置
        mujoco.mj_resetData(model, data)
        if home_key_id >= 0:
            mujoco.mj_resetDataKeyframe(model, data, home_key_id)
        mujoco.mj_forward(model, data)
        if hand_body_id >= 0:
            target_pos = data.xpos[hand_body_id].copy()

# 如果 keyboard 库可用，设置全局键盘监听
if KEYBOARD_AVAILABLE:
    def on_key_event(event):
        key_name = event.name.lower()
        if event.event_type == keyboard.KEY_DOWN:
            keys_pressed[key_name] = True
        elif event.event_type == keyboard.KEY_UP:
            keys_pressed[key_name] = False
    
    keyboard.hook(on_key_event)
    print("✓ 键盘监听已启动（全局监听，无需点击窗口）")
else:
    print("⚠ 未安装 keyboard 库，将尝试使用替代方案")

print("\n" + "="*50)
print("=== 键盘控制说明 ===")
print("W/S: 前后移动 (X轴 +/-)")
print("A/D: 左右移动 (Y轴 -/+)")
print("Q/E: 上下移动 (Z轴 +/-)")
print("R: 重置到初始位置")
print("ESC: 关闭窗口")
print("="*50 + "\n")

# 打开 viewer 窗口
with mujoco.viewer.launch_passive(model, data) as viewer:
    # 设置相机参数
    viewer.cam.lookat[:] = [0.3, 0, 0.4]
    viewer.cam.distance = 2.0
    viewer.cam.azimuth = 120
    viewer.cam.elevation = -20
    
    step = 0
    last_print_time = time.time()
    
    print("窗口已打开，现在可以使用键盘控制机械臂！")
    print("（如果键盘不响应，请确保已安装 keyboard 库）\n")
    
    while viewer.is_running():
        # 更新目标位置（根据键盘输入）
        update_target_from_keys()
        
        # 使用逆运动学求解关节角度
        for _ in range(3):  # 每步迭代几次以提高精度
            solve_ik(model, data, target_pos)
            mujoco.mj_forward(model, data)
        
        # 设置控制命令（位置控制模式，直接使用当前关节位置）
        data.ctrl[:7] = data.qpos[:7]
        data.ctrl[7] = 128  # 夹爪保持中间位置
        
        # 执行仿真步骤
        mujoco.mj_step(model, data)
        
        # 同步 viewer
        viewer.sync()
        
        # 定期打印当前位置
        if time.time() - last_print_time > 0.5:  # 每0.5秒打印一次
            if hand_body_id >= 0:
                current_pos = data.xpos[hand_body_id]
                error = np.linalg.norm(target_pos - current_pos)
                print(f"目标: [{target_pos[0]:.3f}, {target_pos[1]:.3f}, {target_pos[2]:.3f}] | "
                      f"当前位置: [{current_pos[0]:.3f}, {current_pos[1]:.3f}, {current_pos[2]:.3f}] | "
                      f"误差: {error:.4f}m")
            last_print_time = time.time()
        
        step += 1

# 清理
if KEYBOARD_AVAILABLE:
    keyboard.unhook_all()
    print("\n键盘监听已关闭")

In [5]:
# 示例：不使用 viewer，直接控制机器人并获取状态信息
# 重置数据
mujoco.mj_resetData(model, data)

# 设置目标位置（示例：移动到 home 位置）
target_ctrl = np.array([0.0, 0.0, 0.0, -1.57, 0.0, 1.57, -0.785, 128])
data.ctrl[:] = target_ctrl

# 获取 hand body 的 id
hand_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'hand')

# 运行仿真几步
for i in range(100):
    mujoco.mj_step(model, data)
    if i % 20 == 0:
        print(f"Step {i}: 关节位置 = {data.qpos[:7]}")
        if hand_body_id >= 0:
            print(f"        末端执行器位置 = {data.xpos[hand_body_id]}")
        print(f"        夹爪开口 = {data.qpos[7]:.4f} m")
        print()


Step 0: 关节位置 = [-1.13514180e-06 -1.00624918e-04 -1.37459015e-06 -3.00726852e-04
  9.06874162e-07  6.88361060e-04 -2.20464638e-04]
        末端执行器位置 = [0.088 0.    0.926]
        夹爪开口 = 0.0001 m

Step 20: 关节位置 = [ 6.06722942e-05 -4.97444774e-03  7.83162159e-05 -5.50619257e-02
  1.56193227e-04  1.11988952e-01 -4.27252767e-02]
        末端执行器位置 = [1.10140832e-01 2.89047369e-05 9.35074578e-01]
        夹爪开口 = 0.0041 m

Step 40: 关节位置 = [ 1.22440174e-04 -8.48286770e-03  1.57807302e-04 -1.98042376e-01
  3.04733942e-04  2.80086948e-01 -1.24677175e-01]
        末端执行器位置 = [1.64974224e-01 8.01289610e-05 9.43668330e-01]
        夹爪开口 = 0.0090 m

Step 60: 关节位置 = [ 1.44734013e-04 -1.09432559e-02  1.86199457e-04 -4.27549792e-01
  3.89318378e-04  4.67628701e-01 -2.36807085e-01]
        末端执行器位置 = [2.47546841e-01 1.31861122e-04 9.33400648e-01]
        夹爪开口 = 0.0127 m

Step 80: 关节位置 = [ 1.15612266e-04 -1.16940666e-02  1.47879765e-04 -7.40268828e-01
  4.17816844e-04  6.84053020e-01 -3.76853064e-01]
        末端执行器

In [6]:
# 示例：控制机器人执行特定任务
# 比如：打开夹爪，移动手臂，然后闭合夹爪

# 1. 打开夹爪
data.ctrl[7] = 0  # 完全张开
for i in range(50):
    mujoco.mj_step(model, data)

print(f"步骤1: 夹爪已打开，开口 = {data.qpos[7]:.4f} m")

# 2. 移动手臂到新位置
target_joints = np.array([0.5, -0.5, 0.0, -1.0, 0.5, 1.0, -0.5])
data.ctrl[:7] = target_joints
for i in range(100):
    mujoco.mj_step(model, data)
    
print(f"步骤2: 手臂已移动")
print(f"当前关节位置: {data.qpos[:7]}")

# 3. 闭合夹爪
data.ctrl[7] = 255  # 完全闭合
for i in range(50):
    mujoco.mj_step(model, data)
    
print(f"步骤3: 夹爪已闭合，开口 = {data.qpos[7]:.4f} m")


步骤1: 夹爪已打开，开口 = 0.0081 m
步骤2: 手臂已移动
当前关节位置: [ 0.38560371  0.10120423  0.00138491 -1.15478011  0.26381737  1.29824723
 -0.55920621]
步骤3: 夹爪已闭合，开口 = 0.0219 m


## 说明

### 执行器说明：
- **actuator1-7**: 控制7个关节（joint1-7），单位：弧度
- **actuator8**: 控制夹爪，范围 0-255（0=张开，255=闭合）

### 关节限制：
- joint1: -2.8973 到 2.8973 弧度
- joint2: -1.7628 到 1.7628 弧度  
- joint3: -2.8973 到 2.8973 弧度
- joint4: -3.0718 到 -0.0698 弧度
- joint5: -2.8973 到 2.8973 弧度
- joint6: -0.0175 到 3.7525 弧度
- joint7: -2.8973 到 2.8973 弧度

### 使用提示：
1. 在 Jupyter notebook 中使用 `viewer` 会打开新窗口，适合交互式查看
2. 可以使用 `mujoco.mj_step()` 来执行仿真步骤
3. 通过 `data.ctrl` 设置控制命令
4. 通过 `data.qpos` 获取关节位置
5. 通过 `data.xpos`, `data.xquat` 获取末端执行器位置和姿态
